In [ ]:
from cfgnp.train_suite import CheXpertGeneralConfig, CheXpertOODNodeDropConfig
import torch

cfg = CheXpertOODNodeDropConfig(3, True, "chexpert_ood_3", True, size_invariant=True)
model = cfg.model
model.load_state_dict(torch.load("/data/coml-intersection-joins/lina4921/artifacts/graph_model_24-08-2026_22-01.pt"))

# cfg = CheXpertGeneralConfig("chexpert", True)
# model = cfg.model
# model.load_state_dict(torch.load("/data/coml-intersection-joins/lina4921/artifacts//graph_model_18-08-2026_20-09.pt"))

big_config = CheXpertGeneralConfig("chexpert", True)
big_model = big_config.model
model.target_classifiers = big_model.target_classifiers
model.num_features = big_model.num_features
model.max_classes = big_model.max_classes
model.class_counts = big_model.class_counts


In [ ]:
from cfgnp.loss import compute_per_index_metrics_loader
from cfgnp.util.data_paths import ChexpertPath
from cfgnp.graph_approach.train_graph import create_dataset_artificial_graph
from torch_geometric.loader import DataLoader
from cfgnp.util.util import REGRESSION_TARGET_INDICES, DEVICE
import matplotlib.pyplot as plt
import numpy as np
import os

testset = create_dataset_artificial_graph([], cache_path=ChexpertPath.get_graph_ds_path("test"), target_num_nodes=4)

test_loader = DataLoader(testset, batch_size=4, shuffle=False)
model = model.to(device=DEVICE)
model.eval()

generated_images_dir = "/data/coml-intersection-joins/lina4921/results/chexpert_ood3_generated_images"

for intervention_idx, d in enumerate(test_loader):
    orig_image = d.x_orig.reshape((d.batch_size, -1, 3))[:, 4:].reshape((-1, 128, 128, 3))
    # print("Original features ", d.x_orig[:4, 0, 0], ", intervened features ", d.x_int[:4, 0, 0])
    orig_features = d.x_orig[:4, 0, 0]
    int_features = d.x_int[:4, 0, 0]
    dec_orig, _, _ = model.vae.forward(orig_image.to(cfg.device).permute(0, 3, 1, 2))
    dec_img = dec_orig[0].detach().cpu().permute(1, 2, 0).numpy()
    dec_vis = (dec_img - dec_img.min()) / (dec_img.max() - dec_img.min())

    output = model.predict_image(d.to(cfg.device))
    img = output[0].detach().cpu().permute(1, 2, 0).numpy().clip(min=-1, max=1)
    img_vis = (img - img.min()) / (img.max() - img.min())

    # Difference computed on the raw (unnormalized) decoded images so that the two
    # per-image min/max normalizations used for display don't distort the comparison.
    diff = np.abs(img - dec_img).mean(axis=-1)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(dec_vis)
    axes[0].set_title("Original")
    axes[0].axis("off")
    axes[1].imshow(img_vis)
    axes[1].set_title("Counterfactual")
    axes[1].axis("off")
    heatmap = axes[2].imshow(diff, cmap="hot")
    axes[2].set_title("|Factual - Counterfactual|")
    axes[2].axis("off")
    fig.colorbar(heatmap, ax=axes[2], fraction=0.046, pad=0.04)
    fig.tight_layout()
    # plt.show()

    intervention_dir = os.path.join(generated_images_dir, f"counterfactual_{intervention_idx}")
    os.makedirs(intervention_dir, exist_ok=True)
    fig.savefig(os.path.join(intervention_dir, "image.png"))
    plt.close(fig)

    with open(os.path.join(intervention_dir, "features.txt"), "w") as f:
        f.write(f"Original features: {orig_features.tolist()}\n")
        f.write(f"Counterfactual features: {int_features.tolist()}\n")

compute_per_index_metrics_loader(test_loader, model, model.class_counts, REGRESSION_TARGET_INDICES, big_config.target_indice_map, cfg.device)